# Zain Jordan Customer 360 AI Workshop  
## Class 4: Building a RAG Assistant from Telecom Database Content

### Class Goal

In Class 1, we explored the Zain Jordan Customer 360 SQLite database.  
In Class 2, we built a LangChain tools-based customer-care agent.  
In Class 3, we built a natural language SQL agent.  

In Class 4, we will build a **RAG Assistant**.

RAG means **Retrieval-Augmented Generation**.

The idea is simple:

1. Convert useful database rows into text documents.
2. Create embeddings for those documents.
3. Store the embeddings in a vector store.
4. Retrieve relevant documents for a user question.
5. Ask the LLM to answer using only the retrieved context.

---

## Main Class 4 Use Case

### Telecom Plan, Campaign, and Customer Experience RAG Assistant

Example questions:

- Which plan is suitable for a heavy data user?
- Which campaign can be used for prepaid customers?
- What complaint themes appear in the customer experience data?
- Which add-on is suitable for roaming customers?
- What kind of retention message should we draft based on complaints and churn risk?

---

## Important Boundary

This class focuses on:

- RAG basics
- Database rows to documents
- Embeddings
- Vector store
- Retrieval
- RAG answer generation
- RAG as a LangChain tool

We are **not** doing MCP or multi-agent in this class.


# 1. Learning Outcomes

By the end of this notebook, participants should be able to:

1. Explain RAG in simple language.
2. Identify which database tables are suitable for RAG.
3. Convert database rows into text documents.
4. Create embeddings using OpenAI embeddings.
5. Store documents in an in-memory vector store.
6. Retrieve relevant documents using semantic search.
7. Generate answers from retrieved context.
8. Convert RAG into a LangChain tool.
9. Build a simple RAG-powered telecom assistant.


# 2. Class 4 Concept: SQL Agent vs RAG Assistant

| SQL Agent | RAG Assistant |
|---|---|
| Best for structured questions | Best for semantic/explanation questions |
| Generates SQL | Retrieves relevant text chunks |
| Good for counts, ranking, grouping | Good for recommendations and policy-style answers |
| Example: Which city has highest churn? | Example: Which offer suits a roaming customer? |
| Uses tables and columns | Uses documents and context |

### Simple Rule

Use **SQL Agent** when the answer needs calculation.  
Use **RAG** when the answer needs understanding and explanation.


# 3. Install Required Packages

Run this first in Google Colab.

We install:

- `langchain`
- `langchain-openai`
- `langchain-community`
- `langchain-text-splitters`
- `pandas`
- `numpy`


In [ ]:
%pip install -q -U langchain langchain-openai langchain-community langchain-text-splitters pandas numpy


# 4. Import Libraries


In [ ]:
import os
import sqlite3
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Libraries imported successfully.")


# 5. Set OpenAI API Key

In Google Colab:

1. Click the key icon on the left sidebar.
2. Add a secret named `OPENAI_API_KEY`.
3. Paste your OpenAI API key.
4. Enable notebook access for the secret.

This notebook will read the API key from Colab Secrets.


In [ ]:
try:
    from google.colab import userdata
    openai_key = userdata.get("OPENAI_API_KEY")
    if openai_key:
        os.environ["OPENAI_API_KEY"] = openai_key
        print("OpenAI API key loaded from Colab Secrets.")
    else:
        print("OPENAI_API_KEY not found in Colab Secrets.")
except Exception:
    print("Not running in Google Colab, or Colab Secrets not available.")

if not os.environ.get("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY is not set. RAG cells using embeddings/LLM will not run until it is configured.")
else:
    print("OPENAI_API_KEY is available.")


# 6. Upload or Locate the Zain Jordan Database

Upload the same database file used in the earlier classes:

`zain_customer_360_ai_demo.db`

If the file has a slightly different name, this notebook will automatically detect any `.db` file in the current folder.


In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded files:", list(uploaded.keys()))
except Exception:
    print("Google Colab upload is not available here. If running locally, place the .db file in the notebook folder.")


# 7. Connect to the SQLite Database


In [ ]:
db_files = [file for file in os.listdir() if file.endswith(".db")]

if db_files:
    DB_PATH = db_files[0]
else:
    DB_PATH = "zain_customer_360_ai_demo.db"

db_path_obj = Path(DB_PATH).resolve()

print("Database path:", db_path_obj)
print("File exists:", db_path_obj.exists())

conn = sqlite3.connect(str(db_path_obj), check_same_thread=False)

tables_df = pd.read_sql_query("""
SELECT name 
FROM sqlite_master 
WHERE type = 'table'
ORDER BY name;
""", conn)

print("Number of tables:", len(tables_df))
tables_df


# 8. Which Tables Are Good for RAG?

Not every table should become a RAG document.

## Good RAG tables

These contain descriptions, feedback, categories, or business meaning:

- `plans`
- `addons`
- `campaigns`
- `complaints`
- `support_interactions`
- `customer_satisfaction`

## Better for SQL Agent

These are better for calculations, counts, filtering, ranking:

- `customers`
- `subscriptions`
- `invoices`
- `payments`
- `transactions`
- `data_usage_sessions`
- `customer_churn_scores`
- `customer_value_segments`

### Teaching Point

RAG is not a replacement for SQL.  
RAG and SQL solve different problems.


# 9. Preview RAG-Friendly Tables


In [ ]:
for table in ["plans", "addons", "campaigns", "complaints", "support_interactions", "customer_satisfaction"]:
    print(f"\\n--- {table} ---")
    display(pd.read_sql_query(f"SELECT * FROM {table} LIMIT 3;", conn))


# 10. Import LangChain Document Class

A LangChain `Document` usually has:

- `page_content`: the text that will be embedded and searched
- `metadata`: extra information such as source table, row ID, category, etc.


In [ ]:
from langchain_core.documents import Document

print("Document class imported.")


# 11. Helper Function: Create Documents from Plans

We convert each plan row into a readable text document.


In [ ]:
def build_plan_documents(conn):
    df = pd.read_sql_query("""
    SELECT 
        plan_id,
        plan_name,
        plan_category,
        service_type,
        monthly_fee_jod,
        data_allowance_gb,
        local_minutes,
        international_minutes,
        roaming_minutes,
        sms_allowance,
        technology,
        contract_months,
        data_carryover_flag,
        is_business_plan,
        status
    FROM plans
    WHERE status = 'Active';
    """, conn)

    docs = []
    for _, row in df.iterrows():
        text = f"""
Plan ID: {row['plan_id']}
Plan Name: {row['plan_name']}
Category: {row['plan_category']}
Service Type: {row['service_type']}
Monthly Fee: {row['monthly_fee_jod']} JOD
Data Allowance: {row['data_allowance_gb']} GB
Local Minutes: {row['local_minutes']}
International Minutes: {row['international_minutes']}
Roaming Minutes: {row['roaming_minutes']}
SMS Allowance: {row['sms_allowance']}
Technology: {row['technology']}
Contract Months: {row['contract_months']}
Data Carryover: {bool(row['data_carryover_flag'])}
Business Plan: {bool(row['is_business_plan'])}

Use this plan information when recommending telecom plans to customers based on data usage, voice usage, roaming needs, technology preference, price sensitivity, and business or individual requirements.
"""
        docs.append(Document(
            page_content=text.strip(),
            metadata={
                "source_table": "plans",
                "row_id": int(row["plan_id"]),
                "document_type": "plan",
                "plan_name": row["plan_name"],
                "plan_category": row["plan_category"],
            }
        ))
    return docs


plan_docs = build_plan_documents(conn)
print("Plan documents:", len(plan_docs))
print(plan_docs[0].page_content[:1000])


# 12. Helper Function: Create Documents from Add-ons


In [ ]:
def build_addon_documents(conn):
    df = pd.read_sql_query("""
    SELECT 
        addon_id,
        addon_name,
        addon_type,
        price_jod,
        validity_days,
        data_gb,
        minutes,
        sms,
        technology
    FROM addons;
    """, conn)

    docs = []
    for _, row in df.iterrows():
        text = f"""
Add-on ID: {row['addon_id']}
Add-on Name: {row['addon_name']}
Add-on Type: {row['addon_type']}
Price: {row['price_jod']} JOD
Validity Days: {row['validity_days']}
Data: {row['data_gb']} GB
Minutes: {row['minutes']}
SMS: {row['sms']}
Technology: {row['technology']}

Use this add-on information when recommending extra data, roaming, voice, SMS, or technology-specific bundles to customers.
"""
        docs.append(Document(
            page_content=text.strip(),
            metadata={
                "source_table": "addons",
                "row_id": int(row["addon_id"]),
                "document_type": "addon",
                "addon_type": row["addon_type"],
            }
        ))
    return docs


addon_docs = build_addon_documents(conn)
print("Add-on documents:", len(addon_docs))
print(addon_docs[0].page_content[:1000])


# 13. Helper Function: Create Documents from Campaigns


In [ ]:
def build_campaign_documents(conn):
    df = pd.read_sql_query("""
    SELECT 
        campaign_id,
        campaign_name,
        campaign_type,
        start_date,
        end_date,
        target_segment,
        offer_description,
        channel
    FROM campaigns;
    """, conn)

    docs = []
    for _, row in df.iterrows():
        text = f"""
Campaign ID: {row['campaign_id']}
Campaign Name: {row['campaign_name']}
Campaign Type: {row['campaign_type']}
Start Date: {row['start_date']}
End Date: {row['end_date']}
Target Segment: {row['target_segment']}
Offer Description: {row['offer_description']}
Channel: {row['channel']}

Use this campaign information when recommending promotional offers, campaign targeting, channel selection, and customer engagement ideas.
"""
        docs.append(Document(
            page_content=text.strip(),
            metadata={
                "source_table": "campaigns",
                "row_id": int(row["campaign_id"]),
                "document_type": "campaign",
                "campaign_type": row["campaign_type"],
                "target_segment": row["target_segment"],
            }
        ))
    return docs


campaign_docs = build_campaign_documents(conn)
print("Campaign documents:", len(campaign_docs))
print(campaign_docs[0].page_content[:1000])


# 14. Helper Function: Create Documents from Complaints

For training cost control, we limit the number of complaint rows.

You can increase `limit` if needed.


In [ ]:
def build_complaint_documents(conn, limit=200):
    df = pd.read_sql_query("""
    SELECT 
        complaint_id,
        customer_id,
        complaint_date,
        complaint_category,
        complaint_description,
        severity,
        status,
        compensation_amount_jod
    FROM complaints
    ORDER BY complaint_date DESC
    LIMIT ?;
    """, conn, params=(limit,))

    docs = []
    for _, row in df.iterrows():
        text = f"""
Complaint ID: {row['complaint_id']}
Customer ID: {row['customer_id']}
Complaint Date: {row['complaint_date']}
Complaint Category: {row['complaint_category']}
Complaint Description: {row['complaint_description']}
Severity: {row['severity']}
Status: {row['status']}
Compensation Amount: {row['compensation_amount_jod']} JOD

Use this complaint information to identify customer pain points, service issues, complaint themes, and customer experience risks.
"""
        docs.append(Document(
            page_content=text.strip(),
            metadata={
                "source_table": "complaints",
                "row_id": int(row["complaint_id"]),
                "customer_id": int(row["customer_id"]),
                "document_type": "complaint",
                "complaint_category": row["complaint_category"],
                "severity": row["severity"],
            }
        ))
    return docs


complaint_docs = build_complaint_documents(conn, limit=200)
print("Complaint documents:", len(complaint_docs))
print(complaint_docs[0].page_content[:1000])


# 15. Helper Function: Create Documents from Support Interactions


In [ ]:
def build_support_documents(conn, limit=200):
    df = pd.read_sql_query("""
    SELECT 
        interaction_id,
        customer_id,
        interaction_datetime,
        channel,
        reason_category,
        issue_type,
        priority,
        resolution_status,
        resolution_time_minutes,
        customer_sentiment
    FROM support_interactions
    ORDER BY interaction_datetime DESC
    LIMIT ?;
    """, conn, params=(limit,))

    docs = []
    for _, row in df.iterrows():
        text = f"""
Support Interaction ID: {row['interaction_id']}
Customer ID: {row['customer_id']}
Interaction DateTime: {row['interaction_datetime']}
Channel: {row['channel']}
Reason Category: {row['reason_category']}
Issue Type: {row['issue_type']}
Priority: {row['priority']}
Resolution Status: {row['resolution_status']}
Resolution Time Minutes: {row['resolution_time_minutes']}
Customer Sentiment: {row['customer_sentiment']}

Use this support interaction information to understand customer support reasons, sentiment, support channels, issue types, and service quality.
"""
        docs.append(Document(
            page_content=text.strip(),
            metadata={
                "source_table": "support_interactions",
                "row_id": int(row["interaction_id"]),
                "customer_id": int(row["customer_id"]),
                "document_type": "support_interaction",
                "channel": row["channel"],
                "sentiment": row["customer_sentiment"],
            }
        ))
    return docs


support_docs = build_support_documents(conn, limit=200)
print("Support documents:", len(support_docs))
print(support_docs[0].page_content[:1000])


# 16. Helper Function: Create Documents from Customer Satisfaction Feedback


In [ ]:
def build_satisfaction_documents(conn, limit=200):
    df = pd.read_sql_query("""
    SELECT 
        survey_id,
        customer_id,
        survey_date,
        nps_score,
        csat_score,
        feedback_text,
        sentiment
    FROM customer_satisfaction
    ORDER BY survey_date DESC
    LIMIT ?;
    """, conn, params=(limit,))

    docs = []
    for _, row in df.iterrows():
        text = f"""
Survey ID: {row['survey_id']}
Customer ID: {row['customer_id']}
Survey Date: {row['survey_date']}
NPS Score: {row['nps_score']}
CSAT Score: {row['csat_score']}
Feedback Text: {row['feedback_text']}
Sentiment: {row['sentiment']}

Use this satisfaction feedback to understand customer sentiment, common issues, loyalty signals, and improvement opportunities.
"""
        docs.append(Document(
            page_content=text.strip(),
            metadata={
                "source_table": "customer_satisfaction",
                "row_id": int(row["survey_id"]),
                "customer_id": int(row["customer_id"]),
                "document_type": "satisfaction_feedback",
                "sentiment": row["sentiment"],
            }
        ))
    return docs


satisfaction_docs = build_satisfaction_documents(conn, limit=200)
print("Satisfaction documents:", len(satisfaction_docs))
print(satisfaction_docs[0].page_content[:1000])


# 17. Combine All Documents

For this demo, we use a moderate number of documents to control embedding cost.

You can increase or decrease limits depending on time and budget.


In [ ]:
documents = (
    plan_docs
    + addon_docs
    + campaign_docs
    + complaint_docs
    + support_docs
    + satisfaction_docs
)

print("Total documents before splitting:", len(documents))

from collections import Counter
doc_type_counts = Counter(doc.metadata["document_type"] for doc in documents)
doc_type_counts


# 18. Split Documents into Chunks

Even though our rows are short, splitting is still a good habit.

For longer documents, chunking helps retrieval work better.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

split_docs = text_splitter.split_documents(documents)

print("Total chunks after splitting:", len(split_docs))
print(split_docs[0].page_content[:1000])
print(split_docs[0].metadata)


# 19. Create Embeddings and Vector Store

We use:

- `OpenAIEmbeddings` to convert text into vectors
- `InMemoryVectorStore` for a simple classroom-friendly vector store

For production, you may later use a persistent vector database such as Pinecone, Chroma, Weaviate, Milvus, or other options.


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

EMBEDDING_MODEL = "text-embedding-3-small"

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = InMemoryVectorStore(embeddings)

print("Embedding model and vector store initialized.")


# 20. Add Documents to the Vector Store

This step creates embeddings for all chunks.

It may take a little time depending on the number of chunks.


In [ ]:
document_ids = vector_store.add_documents(split_docs)

print("Documents added to vector store:", len(document_ids))


# 21. Test Retrieval

Retrieval means:

> Find the most relevant documents for the user question.


In [ ]:
query = "Which plan or add-on is useful for a customer who needs roaming?"

retrieved_docs = vector_store.similarity_search(query, k=5)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Retrieved Document {i} ---")
    print("Metadata:", doc.metadata)
    print(doc.page_content[:800])
    print()


# 22. Create a Retriever

A retriever is a simple interface that fetches relevant documents for a query.


In [ ]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 5}
)

docs = retriever.invoke("Find campaigns for prepaid users")
for doc in docs:
    print(doc.metadata)
    print(doc.page_content[:500])
    print("-" * 100)


# 23. Create the Chat Model

We use a cost-effective model for classroom demos.

You can change the model based on your access.


In [ ]:
from langchain.chat_models import init_chat_model

MODEL_NAME = "gpt-4.1-mini"

llm = init_chat_model(
    MODEL_NAME,
    model_provider="openai",
    temperature=0
)

print("LLM initialized:", MODEL_NAME)


# 24. Build a Simple RAG Function

This is a clear two-step RAG implementation:

1. Retrieve documents.
2. Generate answer using retrieved context.


In [ ]:
def format_docs(docs):
    formatted = []
    for i, doc in enumerate(docs, start=1):
        source = doc.metadata.get("source_table", "unknown")
        doc_type = doc.metadata.get("document_type", "unknown")
        row_id = doc.metadata.get("row_id", "unknown")

        formatted.append(
            f"[Document {i} | Source: {source} | Type: {doc_type} | Row ID: {row_id}]\\n{doc.page_content}"
        )
    return "\\n\\n".join(formatted)


def rag_answer(question: str, k: int = 5) -> str:
    docs = retriever.invoke(question)
    context = format_docs(docs[:k])

    prompt = f"""
You are a professional telecom AI assistant for Zain Jordan.

Answer the user's question using only the retrieved context below.

Rules:
- Do not invent facts.
- If the context is not enough, say what is missing.
- Keep the answer structured and business-friendly.
- Where useful, include recommended next action.

Retrieved Context:
{context}

User Question:
{question}

Answer:
"""

    response = llm.invoke(prompt)
    return response.content


# 25. Demo 1: Plan Recommendation with RAG


In [ ]:
question = "Which plans are suitable for a customer who wants a lot of mobile data and 5G?"

answer = rag_answer(question)
print(answer)


# 26. Demo 2: Roaming Add-on Recommendation


In [ ]:
question = "A customer is traveling and needs roaming support. Which add-ons or offers seem relevant?"

answer = rag_answer(question)
print(answer)


# 27. Demo 3: Campaign Recommendation


In [ ]:
question = "Which campaigns are suitable for prepaid users?"

answer = rag_answer(question)
print(answer)


# 28. Demo 4: Customer Experience Theme Analysis


In [ ]:
question = "What are the common customer experience issues based on complaints and support interactions?"

answer = rag_answer(question)
print(answer)


# 29. Demo 5: Support Improvement Recommendation


In [ ]:
question = """
Based on the retrieved support interactions and satisfaction feedback,
what should Zain Jordan improve in customer care?
"""

answer = rag_answer(question)
print(answer)


# 30. Add Sources to the Answer

For business use, it is helpful to show which retrieved documents were used.


In [ ]:
def rag_answer_with_sources(question: str, k: int = 5) -> str:
    docs = retriever.invoke(question)[:k]
    context = format_docs(docs)

    prompt = f"""
You are a professional telecom AI assistant for Zain Jordan.

Answer the user's question using only the retrieved context.

Rules:
- Do not invent facts.
- If context is insufficient, say so.
- Keep the answer structured.
- Include a short "Sources Used" section using source table and row ID.

Retrieved Context:
{context}

User Question:
{question}

Answer:
"""
    response = llm.invoke(prompt)

    source_lines = []
    for doc in docs:
        source_lines.append(
            f"- {doc.metadata.get('source_table')} | {doc.metadata.get('document_type')} | Row ID: {doc.metadata.get('row_id')}"
        )

    return response.content + "\\n\\nRetrieved Sources:\\n" + "\\n".join(source_lines)


question = "Recommend a campaign or offer for a customer who needs more data."

answer = rag_answer_with_sources(question)
print(answer)


# 31. Create a RAG Tool

Now we convert RAG into a LangChain tool.

This connects Class 4 back to Class 2.

The agent can use the RAG tool when it needs plan, add-on, campaign, complaint, support, or satisfaction context.


In [ ]:
from langchain.tools import tool

@tool
def search_telecom_knowledge(question: str) -> str:
    """Search Zain Jordan telecom plan, add-on, campaign, complaint, support, and satisfaction knowledge using RAG."""
    return rag_answer_with_sources(question, k=5)


print(search_telecom_knowledge.invoke("Find roaming offers or add-ons."))


# 32. Build a Simple RAG Agent

This agent has one tool: the RAG search tool.

It is useful when users ask recommendation-style questions.


In [ ]:
from langchain.agents import create_agent

rag_agent_system_prompt = """
You are a professional telecom recommendation assistant for Zain Jordan.

Use the search_telecom_knowledge tool when you need information about:
- plans
- add-ons
- campaigns
- complaints
- support interactions
- satisfaction feedback
- customer experience themes

Do not guess.
Use retrieved context.
Keep answers structured, concise, and business-friendly.
"""

rag_agent = create_agent(
    model=llm,
    tools=[search_telecom_knowledge],
    system_prompt=rag_agent_system_prompt,
)

print("RAG agent created successfully.")


# 33. Helper Function to Run the RAG Agent


In [ ]:
def extract_final_text(result):
    last_message = result["messages"][-1]

    if hasattr(last_message, "content") and isinstance(last_message.content, str):
        return last_message.content

    if hasattr(last_message, "content_blocks"):
        parts = []
        for block in last_message.content_blocks:
            if isinstance(block, dict):
                if "text" in block:
                    parts.append(block["text"])
                elif "content" in block:
                    parts.append(str(block["content"]))
                else:
                    parts.append(str(block))
            else:
                parts.append(str(block))
        return "\\n".join(parts)

    return str(last_message)


def run_rag_agent(question: str):
    result = rag_agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })
    return extract_final_text(result)


# 34. Demo 6: RAG Agent Question


In [ ]:
question = """
A customer wants better data and may respond to a campaign.
Use the telecom knowledge base to recommend a suitable plan, add-on, or campaign.
"""

answer = run_rag_agent(question)
print(answer)


# 35. Demo 7: Customer Experience RAG Agent


In [ ]:
question = """
What customer experience problems appear in the knowledge base, and what action should the customer-care team take?
"""

answer = run_rag_agent(question)
print(answer)


# 36. Optional: Combine SQL + RAG Manually

This is not yet a full multi-agent workflow.

But we can manually combine:

1. SQL to get customer usage or churn.
2. RAG to recommend plan/add-on/campaign.

This prepares participants for later multi-agent and capstone work.


In [ ]:
def get_customer_usage_for_recommendation(customer_id: int) -> str:
    query = """
    SELECT 
        c.customer_id,
        c.full_name,
        c.city,
        c.customer_segment,
        s.service_type,
        p.plan_name,
        p.monthly_fee_jod,
        p.data_allowance_gb,
        COUNT(d.session_id) AS data_sessions,
        ROUND(SUM(d.data_used_mb) / 1024.0, 2) AS total_data_used_gb
    FROM customers c
    JOIN subscriptions s
        ON c.customer_id = s.customer_id
    JOIN plans p
        ON s.plan_id = p.plan_id
    LEFT JOIN data_usage_sessions d
        ON s.subscription_id = d.subscription_id
    WHERE c.customer_id = ?
    GROUP BY 
        c.customer_id,
        c.full_name,
        c.city,
        c.customer_segment,
        s.service_type,
        p.plan_name,
        p.monthly_fee_jod,
        p.data_allowance_gb
    ORDER BY total_data_used_gb DESC;
    """
    df = pd.read_sql_query(query, conn, params=(customer_id,))
    if df.empty:
        return "No customer usage data found."
    return df.to_string(index=False)


customer_context = get_customer_usage_for_recommendation(42)
print(customer_context)


In [ ]:
question = f"""
Here is a customer usage summary:

{customer_context}

Based on the available Zain Jordan plans, add-ons, and campaigns in the RAG knowledge base,
recommend a suitable plan, add-on, or campaign.
"""

answer = rag_answer_with_sources(question)
print(answer)


# 37. Participant Exercise 1: Test Retrieval

Try different retrieval questions:

1. Find offers for prepaid users.
2. Find add-ons for roaming customers.
3. Find plans with high data allowance.
4. Find common complaints about slow internet.
5. Find customer satisfaction problems.


In [ ]:
test_query = "Find common complaints about slow internet."

retrieved_docs = retriever.invoke(test_query)

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"--- Document {i} ---")
    print(doc.metadata)
    print(doc.page_content[:600])
    print()


# 38. Participant Exercise 2: Ask RAG Questions

Try:

1. Which plan is good for heavy data users?
2. Which offer is suitable for prepaid users?
3. What should we do for customers complaining about slow internet?
4. What are common support issues?
5. What customer-care improvement is suggested by feedback?


In [ ]:
exercise_question = "What should Zain Jordan do for customers complaining about slow internet?"

answer = rag_answer_with_sources(exercise_question)
print(answer)


# 39. Participant Exercise 3: Capstone Thinking

Ask the RAG assistant:

> Suggest three AI application ideas using this RAG knowledge base.

Then discuss which ideas need:

- only RAG
- only SQL
- both SQL and RAG
- later multi-agent/MCP


In [ ]:
capstone_question = """
Suggest three capstone project ideas using this RAG knowledge base.
For each idea, mention:
1. Target user
2. Problem solved
3. Data used
4. Expected output
"""

answer = rag_answer(capstone_question)
print(answer)


# 40. Common RAG Mistakes

## Mistake 1: Putting everything into RAG

Not every table should become documents.

Use SQL for calculations.  
Use RAG for explanations and recommendations.

## Mistake 2: Weak document text

Bad document text gives weak retrieval.

Good documents should explain the row clearly.

## Mistake 3: No source tracking

Always keep metadata such as:

- source table
- row ID
- document type
- category

## Mistake 4: Trusting the LLM without context

RAG should answer from retrieved context, not memory.

## Mistake 5: Too many chunks

For training, keep chunks limited to control cost and speed.


# 41. What We Built Today

In Class 4, we built:

1. A RAG knowledge base from Zain Jordan database rows.
2. Documents from plans, add-ons, campaigns, complaints, support interactions, and satisfaction feedback.
3. Embeddings using OpenAI embeddings.
4. An in-memory vector store.
5. Semantic search / retrieval.
6. A two-step RAG answer function.
7. RAG answers with sources.
8. A RAG tool.
9. A simple RAG agent.
10. A manual SQL + RAG recommendation pattern.

---

## Next Class

In the next class, we will build a **Multi-Agent Customer Care Copilot**.

That system can include:

- Customer Data Agent
- SQL Analysis Agent
- RAG Recommendation Agent
- Communication Agent

Later, we will expose selected tools through MCP.


# 42. Trainer Closing Script

Today, we learned how to build RAG from telecom database content.

The key idea is:

> RAG lets the AI search relevant context before answering.

We converted database rows into documents, embedded them, stored them in a vector store, retrieved relevant context, and generated business-friendly answers.

This is useful for:

- plan recommendations
- add-on recommendations
- campaign suggestions
- complaint understanding
- support improvement ideas
- customer experience insights

In the next class, we will combine these ideas into a multi-agent customer-care copilot.
